# Sales Forecasting: Time-Series Analysis (Rossmann Store Sales)

An end-to-end pipeline: ingestion -> EDA -> feature engineering -> SARIMA / LightGBM / LSTM modelling -> evaluation -> multi-horizon forecasts with confidence bands -> report.

This notebook walks through one store end to end. The same steps run for multiple stores via `python run_pipeline.py`.

## 0. Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '..')          # project root
sys.path.insert(0, '../src')      # package
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sales_forecast.config import Config
from sales_forecast.ingestion import load_data, prepare_store_data
from sales_forecast.features import build_features, GBM_FEATURE_COLS, LSTM_FEATURE_COLS
from sales_forecast.eda import run_eda
from sales_forecast.evaluate import compute_metrics, error_segmentation
from sales_forecast.models import SarimaForecaster, GbmForecaster, LstmForecaster
from sales_forecast.forecast import build_future_frame, recursive_gbm_forecast, forecast_table
from sales_forecast.pipeline import run_store, forecast_sales
from sales_forecast.report import write_report

cfg = Config()
print('Config OK. Stores:', cfg.stores, '| Horizons:', cfg.horizons)

## 1. Data ingestion & cleaning

In [ ]:
train, store = load_data(cfg)
print('train shape:', train.shape, '| store metadata:', store.shape)
train[['Date', 'Sales', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']].head(5)

In [ ]:
df = prepare_store_data(train, store, 1097)
df = build_features(df, store[store['Store'] == 1097].iloc[0], cfg)
print('engineered frame:', df.shape)
print('date range:', df['Date'].min().date(), '->', df['Date'].max().date())
print('missing timestamps:', int(df['Date'].duplicated().sum()))

## 2. Exploratory data analysis

Time series, weekly/monthly seasonality, promo effect, ACF/PACF, correlation heatmap and seasonal decomposition.

In [ ]:
paths = run_eda(df, 1097, cfg)
print('EDA figures written:', len(paths))

In [ ]:
dow = df.groupby('day_of_week')['Sales'].mean()
names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
print('Avg sales by weekday:', {names[i]: round(v) for i, v in dow.items()})
p0 = df[df['Promo']==0]['Sales'].mean(); p1 = df[df['Promo']==1]['Sales'].mean()
print(f'Promo lift: +{100*(p1/p0-1):.1f}%')

## 3. Feature engineering

Calendar/cyclical features, holiday and promo encodings, store metadata, and lag/rolling statistics of the log-sales series.

In [ ]:
print('GBM features (%d):' % len(GBM_FEATURE_COLS))
print(', '.join(GBM_FEATURE_COLS))
print()
print('LSTM window features (%d):' % len(LSTM_FEATURE_COLS))
print(', '.join(LSTM_FEATURE_COLS))

## 4. Modelling

Chronological 80/20 split. Three models:
- **SARIMA** (classical, pmdarima auto_arima, period 7)
- **LightGBM** (gradient boosting on lag/rolling features, time-series CV)
- **LSTM** (PyTorch sequence model, recursive multi-step)

The cell below runs the full per-store pipeline (about 3 minutes).

In [ ]:
res = run_store(1097, cfg)
for k, v in res['metrics'].items():
    if 'recursive' in k:
        print(f"{k:24s} MAE={v['MAE']:8.0f} RMSE={v['RMSE']:8.0f} "
              f"MAPE={v['MAPE']:5.2f}%  MASE={v['MASE']:.3f}")
print('\nBest model by recursive-test RMSE:', res['best_model'])

## 5. Evaluation

Metrics: MAE, RMSE, MAPE, MASE. Recursive forecasts respect the chronological split (first 80% train, last 20% test).

In [ ]:
best = res['best_model']
print('Test-period recursive metrics for best model (%s):' % best)
print(pd.Series(res['metrics'][best + '_recursive']))
print('\nTop 10 LightGBM features:')
print(res['feature_importance'].head(10).to_string())

## 6. Multi-horizon forecasts (30 / 90 / 180 days)

In [ ]:
fc30 = res['forecasts'][30][best]
fc30.head(10)

In [ ]:
out = cfg.forecasts_dir / f'store_1097_30d_{best.lower()}.csv'
fc30.to_csv(out, index=False)
print('saved:', out)

## 7. Public function interface

`forecast_sales(store_id, horizon)` returns the deliverable forecast dataframe: `date, predicted_sales, lower_80, upper_80, lower_95, upper_95`.

In [ ]:
fc = forecast_sales(1097, horizon=30)
fc.head()

## 8. Full multi-store pipeline & report

Run all configured stores and regenerate `outputs/report.md`. (This cell can take several minutes; the per-store results above are already cached in `outputs/`.)

In [ ]:
from sales_forecast.pipeline import run_pipeline
all_results = run_pipeline(cfg)
report_path = write_report(all_results, cfg)
print('Report:', report_path)

## Summary

- **LightGBM** with lag/rolling features is the most accurate multi-step forecaster (recursive-test MAPE 7-8%).
- The **LSTM** is excellent one-step-ahead (MAPE ~1%) but compounds errors over long recursive horizons.
- **SARIMA** provides a solid classical baseline and native prediction intervals.
- Forecast CSVs, model artifacts and `report.md` live under `outputs/`.